# Семинар 3. Аудит сырых данных ПРАЙМ

Легенда:
В понедельник в отчёт уходят четыре числа: клиенты по каналам привлечения,
средняя длительность сессии, оборот в партнёрских сервисах, выручка от подписки.
Все четыре посчитаны, все четыре выглядят нормально.

**Все четыре неверны.**

За пару мы найдём четыре дефекта и померяем, на сколько каждый сдвигает своё
число. Сдвиги будут крошечные — от 0,11 % до 2 %. Ни один не виден на графике.

Добавил крч ещё метки, чтобы понимать что более важно, что менее важно

| Метка | Что значит |
|---|---|
| **понимать** | спросят на защите и на проверочной |
| **уметь писать** | понадобится в ДЗ-2 и в проекте |
| **знать, что существует** | достаточно помнить, что такое бывает, и загуглить |


## Выгрузка данных в пандас табличку


In [ ]:
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py:
# она читает строку подключения из .env, поэтому пароля здесь нет.
from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # День события считаем по UTC — так он у всех одинаковый.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        # Сервер общий на всю группу, соединения не копим.
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])


In [ ]:
# Стоит ли pyarrow — от этого зависят ваши числа памяти.
try:
    import pyarrow

    print("pyarrow стоит, версия", pyarrow.__version__)
    print("Ваши числа памяти будут меньше моих — но не вдвое: uuid всё равно приезжают объектами.")
except ModuleNotFoundError:
    print("pyarrow не стоит — как и задумано, числа совпадут с моими.")

print("pandas", pd.__version__)


In [ ]:
# Берём срез: двадцать регионов из восьмидесяти девяти. Так и работают с большой
# таблицей — не «выгружу всё, потом посмотрю».
# Если тянется дольше двух минут, замените 20 на 5 и перезапустите.
clients = q("""
    select client_id, registered_at, region_code, time_zone,
           birth_year, acquisition_channel, kyc_level
    from prime.clients
    where region_code <= 20
    order by client_id
""")

print(len(clients), "строк")
clients.head(3)


---

## 1. Приёмка таблицы: семь вопросов

Это те самые семь вопросов со слайда 27 лекции. Их задают **любой** таблице до
того, как что-то по ней считать — своей, чужой, выгруженной коллегой.

1. Сколько строк — и сколько я ждал?
2. Какие типы выбрал за меня pandas?
3. Сколько таблица весит?
4. Одна строка — один кто? Нет ли дублей?
5. Где пропуски?
6. Нет ли значений, которых не может быть, и противоречий между колонками?
7. Какой период покрыт и нет ли дыр?

> **Зачем это в работе.** Выгрузку вам пришлют по почте, без описания, и спросят
> число к завтрашнему утру. Семь вопросов — это первые десять минут работы с ней.


In [ ]:
# Вопросы 1 и 2: сколько строк и какие типы выбрал pandas.
print("строк:", len(clients))
print("регионов:", clients["region_code"].nunique())   # 20 — значит where доехал
clients.dtypes


In [ ]:
# Вопрос 3: сколько таблица весит.
# Без одного аргумента pandas покажет 6,3 МБ. Это неправда.
print(clients.memory_usage(deep=ЗАПОЛНИТЕ).sum() / 1e6, "МБ")


**25,2 МБ против 6,3.** Без `deep=True` pandas считает только сами ячейки колонки:
у текстовой колонки это указатели по 8 байт, а сами строки лежат где-то ещё, и их
он не смотрит.

Куда ушли 25 мегабайт:

| Колонка | МБ | Что это |
|---|---:|---|
| `client_id` | 8,09 | uuid, объект по 72 байта каждый |
| `time_zone` | 7,04 | `Europe/Moscow` — текст, одиннадцать разных значений |
| `acquisition_channel` | 6,43 | шесть разных значений |
| четыре числовые | по 0,90 | `int64` и `datetime64` |

Полная `clients` — **111,9 МБ**, и там та же картина.

**Знать, что существует.** `df.info()` напечатает то же число как «24.0 MB» — это
мебибайты, 2²⁰ байт. Одно и то же число в двух единицах.


In [ ]:
# Шесть значений на сто двенадцать тысяч строк. Как хранить, чтобы не хранить
# слово «performance» сто тысяч раз?
before = clients["acquisition_channel"].memory_usage(deep=True) / 1e6
after = clients["acquisition_channel"].astype(ЗАПОЛНИТЕ).memory_usage(deep=True) / 1e6

print(f"{before:.2f} МБ -> {after:.2f} МБ, в {before / after:.0f} раз легче")


In [ ]:
# А теперь то же самое с ключом.
before = clients["client_id"].memory_usage(deep=True) / 1e6
after = clients["client_id"].astype("category").memory_usage(deep=True) / 1e6

print(f"{before:.2f} МБ -> {after:.2f} МБ")
print(f"стало ХУЖЕ на {100 * (after - before) / before:.1f} %")


**`category` — не панацея.**

Если: `nunique() / len()`. меньше 0.1 – категория, иначе оставляйте как есть

> Работает на любом объёме: и на ста тысячах строк, и на пятистах — те же
> плюс пять процентов, потому что это ровно четыре лишних байта на строку.

**Уметь писать.** `astype("category")` для колонок с коротким списком значений.

И главное, ради чего весь раздел: **нужной нам информации здесь на полмегабайта,
а привезли мы двадцать пять.** Список колонок в `select` дешевле любого `astype`.


---

## 2. Клиенты по каналам привлечения

Маркетинг просит разбивку: сколько клиентов пришло из каждого канала.
Одна строка кода.


In [ ]:
# Один аргумент решает, увидите вы дефект или нет.
channels = clients["acquisition_channel"].value_counts(dropna=ЗАПОЛНИТЕ)

print(channels)
print("сумма по каналам:", channels.sum(), "  строк в таблице:", len(clients))


In [ ]:
# Спутник любой группировки: где пропуски и сколько их.
missing = clients.isna().sum()
print(missing[missing > 0])

bad = clients["acquisition_channel"].isna().sum()
print(f"\nбез канала: {bad} строк, это {100 * bad / len(clients):.2f} % таблицы")


**Две тысячи двести восемьдесят два клиента не попали ни в одну строку отчёта.**
По умолчанию `value_counts` выбрасывает пропуски — и не говорит об этом. То же
самое делает `groupby`: ключ со значением `NaN` просто исчезает.

На полной таблице это **десять тысяч человек** из пятисот тысяч, ровно те же 2 %.

Что с ними делать — вопрос не к нам. Можно отнести в «неизвестно», можно
восстановить по другим признакам, можно выкинуть и честно об этом написать. Выбор
между этими тремя — разговор на «Методах анализа данных». **Наше дело инженерное:
знать точное число и точный сдвиг, который оно даёт.**

**Уметь писать.** `dropna=False` в `value_counts` и в `groupby` — и `isna().sum()`
рядом с каждой группировкой.


---

## 3. Оборот в партнёрских сервисах

Дальше — таблица покупок. Берём июль, две колонки: сумма и статус.


In [ ]:
tx = q("""
    select amount, status
    from prime.transactions
    where occurred_at >= '2026-07-01' and occurred_at < '2026-08-01'
    order by transaction_id
""")

print(len(tx), "строк")
tx["amount"].describe()


In [ ]:
# Какая сумма покупки невозможна? Ноль — это невозможно или это подарок?
# Где ставим границу — и почему именно там?
bad = tx[tx["amount"] ЗАПОЛНИТЕ 0]

print("подозрительных строк:", len(bad))
print("из них отрицательных:", (tx["amount"] < 0).sum())
print("из них нулевых:      ", (tx["amount"] == 0).sum())


In [ ]:
ok = tx["status"] == "success"

before = tx.loc[ok, "amount"].sum()
after = tx.loc[ok & (tx["amount"] > 0), "amount"].sum()

print(f"оборот как есть:  {before:>15,.2f} ₽".replace(",", " "))
print(f"оборот без мусора:{after:>15,.2f} ₽".replace(",", " "))
print(f"разница:          {after - before:>+15,.2f} ₽".replace(",", " "))


**Убрали 0,3 % строк — и оборот вырос на восемьсот тринадцать тысяч.**

Это стоит подержать в голове секунду. Обычно чистка данных уменьшает числа: мы
выбрасываем строки, сумма падает. Здесь наоборот, потому что мусор был **со
знаком**: шестьсот девятнадцать покупок с отрицательной суммой тихо вычитались из
оборота.

Дефект, который увеличивает метрику, опаснее дефекта, который её уменьшает: никто
не приходит разбираться, когда цифра выросла.

> **Зачем это в работе.** «Почему у нас оборот меньше, чем в кассе» — типовой
> вопрос от финансов. В половине случаев ответ ровно такой.


---

## 4. Средняя длительность сессии


In [ ]:
su = q("""
    select started_at, ended_at
    from prime.service_usage
    where started_at >= '2026-07-01' and started_at < '2026-08-01'
    order by usage_id
""")

reversed_rows = su["ended_at"] < su["started_at"]
minutes = (su["ended_at"] - su["started_at"]).dt.total_seconds() / 60

print(f"строк:                 {len(su)}")
print(f"кончились до начала:   {reversed_rows.sum()}")
print(f"средняя как есть:      {minutes.mean():.2f} мин")
print(f"средняя без них:       {minutes[~reversed_rows].mean():.2f} мин")
print(f"медиана как есть:      {minutes.median():.2f} мин")
print(f"медиана без них:       {minutes[~reversed_rows].median():.2f} мин")


Инвариант здесь связывает **две колонки**: `ended_at >= started_at`. Ни одна из них
по отдельности не выглядит странно — неправдоподобна только пара.

Средняя сдвинулась на 0,11 минуты, медиана почти не двинулась: 17,97 → 18,00.
Медиана устойчива к выбросам, и это её главное свойство — но заметьте, что
**устойчивость работает против нас**: дефект в данных остался, а прибор его
не показал.


---

## 5. Выручка от подписки

Это ядро пары. Дальше — самый дорогой дефект из четырёх, и самый незаметный.


In [ ]:
payments = q("""
    select payment_id, subscription_id, paid_at, amount, status
    from prime.payments
    where paid_at >= '2026-07-01' and paid_at < '2026-08-01'
    order by payment_id
""")

revenue = payments.loc[payments["status"] == "success", "amount"].sum()

print(f"строк:     {len(payments)}")
print(f"успешных:  {(payments['status'] == 'success').sum()}")
print(f"выручка:   {revenue:,.2f} ₽".replace(",", " "))
payments.head(3)


In [ ]:
# Два платежа — это один и тот же платёж. По каким колонкам это видно?
# Назовите список.
dups = payments.duplicated(subset=[ЗАПОЛНИТЕ])

print("дублей:", dups.sum())


**Шесть с половиной тысяч.** Это три процента всех платежей июля — и если сейчас
их выбросить, выручка упадёт заметно.

По всей истории тот же способ находит **2 665 896 дублей из 3 147 409 платежей**.
Восемьдесят пять процентов таблицы.

Вопрос, прежде чем мы пойдём дальше: **сколько списаний по одной подписке за месяц
бывает в норме?**


In [ ]:
# А если добавить в список момент списания? Сколько теперь?
dups_strict = payments.duplicated(subset=["subscription_id", "amount", ЗАПОЛНИТЕ])

print("дублей:", dups_strict.sum())


Два крайних ответа, и оба неверны.

Без времени «дубль» — это просто **следующий месяц**: подписка списывает одну
и ту же сумму каждые тридцать дней, и `duplicated` честно метит все списания, кроме
первого. Со временем «дубль» — это буквально одна и та же микросекунда, а двойное
списание так не выглядит: платёжный шлюз повторяет запрос через секунды.

**Дубль — это не равенство и не различие. Это близость во времени.**

**Понимать.** Поэтому `drop_duplicates` здесь не помогает ни в каком виде: она
умеет только «совпало точно» и «не совпало».


In [ ]:
# Сортируем и смотрим, сколько прошло с предыдущего платежа
# по той же подписке на ту же сумму.
s = payments.sort_values(["subscription_id", "amount", "paid_at"])
gap = s.groupby(["subscription_id", "amount"])["paid_at"].diff()

print(gap.describe())


In [ ]:
# Два списания по одной подписке на одну сумму.
# Через сколько минут это ещё дубль, а через сколько — уже следующий месяц?
is_dup = gap < pd.Timedelta(minutes=ЗАПОЛНИТЕ)

print("настоящих дублей:", is_dup.sum())
print("подписок затронуто:", s.loc[is_dup, "subscription_id"].nunique())


In [ ]:
# «Двадцать минут» — это с потолка? Проверим, насколько ответ зависит от порога.
for m in (1, 2, 5, 10, 15, 20, 60, 180, 1440):
    print(f"{m:>5} мин -> {(gap < pd.Timedelta(minutes=m)).sum():>4}")


**С пятнадцати минут и до суток ответ один и тот же — 232.**

Это и есть оправдание порога. Не «мне кажется, двадцать» — а «в широком диапазоне
ответ не меняется, значит, он не зависит от моего вкуса». Если бы число ползло
с каждой минутой, порог пришлось бы обсуждать с бизнесом, а не выбирать самому.

**Знать, что существует.** Такая картинка называется анализом чувствительности:
меняем параметр, смотрим, где ответ устойчив.


In [ ]:
clean = s.loc[~is_dup & (s["status"] == "success"), "amount"].sum()

print(f"выручка как есть:  {revenue:>15,.2f} ₽".replace(",", " "))
print(f"выручка без дублей:{clean:>15,.2f} ₽".replace(",", " "))
print(f"сдвиг: {100 * (revenue - clean) / revenue:.3f} %")


---

### Ноль целых одиннадцать сотых процента

Самый маленький сдвиг из четырёх. На графике его не видно. В отчёте он не изменит
ни одного вывода.

А теперь посмотрите на то же число с другой стороны: **232 человека, у которых
в июле дважды списали деньги.** Это 232 обращения в поддержку, 232 возврата,
чей-то испорченный вечер и строчка в отзывах.

| Дефект | Сдвиг метрики | Цена |
|---|---:|---|
| Пропуски в канале | 2,0 % | неверный отчёт маркетингу |
| Суммы ≤ 0 | 0,15 % | оборот занижен |
| Перевёрнутые сессии | 0,4 % | средняя чуть смещена |
| **Двойные списания** | **0,11 %** | **232 человека, с которых взяли дважды** |

**Величина сдвига ничего не говорит о цене дефекта.** Это главное, что стоит
унести с сегодняшней пары.


---

## 6. Проверки переезжают в код

Четыре находки за пару — это хорошо ровно один раз. Через неделю данные обновятся,
и всё придётся искать заново. Поэтому проверки переезжают в функции, а функции —
в ваш репозиторий, в файл `src/prime/checks.py`.

**Форма у всех четырёх одна: функция принимает кадр и возвращает плохие строки.**
Не число, не `True`/`False` — именно строки. Числу вы не сможете задать вопрос
«а покажи их», а строкам сможете.

> **В ДЗ-2 за это 0,5 балла:** одна функция отсюда импортируется в ноутбук задания
> и применяется к собранной витрине. Пустой результат — тоже результат, и он тоже
> приносит полный балл.


In [ ]:
def find_missing_channel(df: pd.DataFrame) -> pd.DataFrame:
    """Клиенты без канала привлечения."""
    return df[df["acquisition_channel"].isna()]


def find_reversed_sessions(df: pd.DataFrame) -> pd.DataFrame:
    """Сессии, которые кончились раньше, чем начались."""
    return df[df["ended_at"] < df["started_at"]]


def find_duplicate_payments(df: pd.DataFrame, gap_minutes: int = 20) -> pd.DataFrame:
    """Повторные списания по одной подписке на одну сумму в пределах gap_minutes."""
    s = df.sort_values(["subscription_id", "amount", "paid_at"])
    gap = s.groupby(["subscription_id", "amount"])["paid_at"].diff()
    return s[gap < pd.Timedelta(minutes=gap_minutes)]


print("три есть, четвёртую пишете вы")


In [ ]:
def find_nonpositive_amounts(df: pd.DataFrame) -> pd.DataFrame:
    """Покупки с невозможной суммой."""
    return ЗАПОЛНИТЕ


print("нашлось:", len(find_nonpositive_amounts(tx)))


In [ ]:
report = pd.DataFrame([
    {"проверка": "нет канала привлечения",
     "строк": len(find_missing_channel(clients)),
     "из скольких": len(clients),
     "сдвиг метрики": "2,03 % клиентов не в отчёте"},
    {"проверка": "сумма <= 0",
     "строк": len(find_nonpositive_amounts(tx)),
     "из скольких": len(tx),
     "сдвиг метрики": "+813 992 ₽ к обороту"},
    {"проверка": "сессия кончилась до начала",
     "строк": len(find_reversed_sessions(su)),
     "из скольких": len(su),
     "сдвиг метрики": "26,88 -> 26,99 мин"},
    {"проверка": "двойное списание",
     "строк": len(find_duplicate_payments(payments)),
     "из скольких": len(payments),
     "сдвиг метрики": "0,112 % выручки, 232 человека"},
])
report["доля"] = (100 * report["строк"] / report["из скольких"]).round(3)

report[["проверка", "строк", "из скольких", "доля", "сдвиг метрики"]]


**Как перенести к себе.** Создайте файл `src/prime/checks.py`, скопируйте туда все
четыре функции и строку `import pandas as pd` сверху. После этого в любом ноутбуке
внутри `prime-monitor` работает:

```python
from prime.checks import find_duplicate_payments
```

Коммит — в `main`, одной строкой: `git add src/prime/checks.py && git commit -m
"проверки данных с семинара 3"`. Файл понадобится в ДЗ-2 и в проекте.


---

## 7. Ваш срез — дальше сами

За каждым логином закреплена своя таблица, свой признак и свой период — те же, что
в ДЗ-1. Задача одна: **посчитать, сколько строк вашего среза нарушают ваш
инвариант.** Одно число, одна-две строки кода. Никаких выводов писать не нужно.

| Ваша таблица | Ваш инвариант | Колонка с датой |
|---|---|---|
| `payments` | нет двойных списаний (порог 20 минут) | `paid_at` |
| `transactions` | сумма покупки больше нуля | `occurred_at` |
| `service_usage` | сессия кончилась не раньше, чем началась | `started_at` |
| `support_tickets` | обращение закрыто не раньше, чем создано | `created_at` |

> **Проверка, которая ничего не нашла, — тоже результат.** У части из вас
> правильный ответ — ноль. Это не значит, что вы что-то сделали не так: это значит,
> что в вашей таблице такого дефекта нет, и вы теперь это знаете, а не
> предполагаете.


In [ ]:
# Четыре места. Возьмите значения из своей строки prime.assignments
# (она же приходила в ДЗ-1).
mine = q("""
    select *
    from prime.ЗАПОЛНИТЕ
    where ЗАПОЛНИТЕ = 'ЗАПОЛНИТЕ'
      and ЗАПОЛНИТЕ >= '2026-07-01' and ЗАПОЛНИТЕ < '2026-08-01'
""")

print(len(mine), "строк")
mine.head(3)


In [ ]:
# Сколько строк вашего среза нарушают ваш инвариант?
# Используйте свою функцию find_* — она уже написана.
violations = None

print("нарушений:", violations)


---

## 8. Задачи со звёздочкой

Дальше — то, что на лекции было на экране, а руками вы этого не делали.
**Эти задачи не сдаются и не оцениваются.**

Смысл в другом: каждая из пяти — это отдельная ловушка pandas, на которой
спотыкаются в реальной работе, и в проекте вы встретите как минимум две. Прорешали
здесь — узнаете в лицо.

Проверка встроена: она знает правильный ответ, но не показывает его. Замените
`None` на своё решение и запустите ячейку целиком. Формат ответа задан в условии —
без него хеш не сойдётся.


In [ ]:
# Это просто запустите
ANSWERS = {
    "★1": "a07d3412ef",
    "★2": "17c4b27ce8",
    "★3": "b9c8c919e5",
    "★4": "a512cfedff",
    "★5": "cdac4b460d"
}


In [ ]:
# Это тоже)
import hashlib

import numpy as np
import pandas as pd


def _norm(x):
    if isinstance(x, bool):
        return repr(x)
    if isinstance(x, float):
        return f"{x:.2f}"
    if isinstance(x, int):
        return str(x)
    if isinstance(x, dict):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_norm(v) for v in x)) + "}"
    if isinstance(x, (list, tuple)):
        return "[" + ",".join(_norm(v) for v in x) + "]"
    if isinstance(x, np.integer):
        return str(int(x))
    if isinstance(x, np.bool_):
        return repr(bool(x))
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    if isinstance(x, pd.Timedelta):
        return str(int(x.total_seconds()))
    if isinstance(x, pd.Index):
        return _norm(list(x))
    if isinstance(x, pd.Series):
        return "{" + ",".join(f"{_norm(k)}:{_norm(v)}" for k, v in sorted(x.items(), key=lambda kv: _norm(kv[0]))) + "}"
    if isinstance(x, pd.DataFrame):
        d = x if isinstance(x.index, pd.RangeIndex) else x.reset_index()
        cols = sorted(map(str, d.columns))
        rows = sorted(tuple(_norm(r[c]) for c in cols) for _, r in d.iterrows())
        return "[" + ",".join("[" + ",".join(r) + "]" for r in rows) + "]"
    return repr(x)


def _show(x) -> str:
    """Короткий показ ответа: кадр на сто тысяч строк в вывод не печатаем."""
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return f"{type(x).__name__} {x.shape}\n{x.head().to_string()}"
    return repr(x)


def check(task: str, got) -> None:
    """Сверить ответ с эталоном. Эталоны лежат хешами — подсмотреть нельзя."""
    if got is None:
        print(f"{task}: не решено")
        return
    digest = hashlib.sha256(_norm(got).encode()).hexdigest()[:10]
    if not ANSWERS:
        print(f"HASH:{task}={digest}")
    elif digest == ANSWERS.get(task):
        print(f"{task}: верно")
    else:
        print(f"{task}: не сходится, получилось {_show(got)}")


print("проверка готова")


### ★ 1. Индекс складывает за вас

Слайд 8 лекции. Две выручки по дням: первая половина июля и вторая декада —
окна перекрываются, но не совпадают.

```python
a = (payments[payments["status"] == "success"]
     .assign(day=lambda d: d["paid_at"].dt.date)
     .query("'2026-07-01' <= day.astype('str') <= '2026-07-15'")
     .groupby("day")["amount"].sum())
b = (payments[payments["status"] == "success"]
     .assign(day=lambda d: d["paid_at"].dt.date)
     .query("'2026-07-10' <= day.astype('str') <= '2026-07-25'")
     .groupby("day")["amount"].sum())
```

Сложите их двумя способами: обычным `a + b` и через `a.add(b, fill_value=0)`.

**Ответ:** сколько рублей теряет обычное сложение. Одно число, два знака после
запятой: `round(правильное - наивное, 2)`.


In [ ]:
ok = payments[payments["status"] == "success"].assign(day=lambda d: d["paid_at"].dt.date)
a = ok[(ok["day"] >= pd.Timestamp("2026-07-01").date()) & (ok["day"] <= pd.Timestamp("2026-07-15").date())].groupby("day")["amount"].sum()
b = ok[(ok["day"] >= pd.Timestamp("2026-07-10").date()) & (ok["day"] <= pd.Timestamp("2026-07-25").date())].groupby("day")["amount"].sum()

answer = None   # ← ваш расчёт

check("★1", answer)


### ★ 2. Метка — не номер строки

Слайды 7 и 18. Отфильтруйте `clients` по каналу `organic` — получится кадр,
у которого индекс остался от исходной таблицы: 3, 7, 11 и так далее.

Возьмите из него два среза: `f.loc[0:10]` и `f.iloc[0:10]`.

**Ответ:** список из двух чисел — длина первого среза и длина второго,
`[len(по меткам), len(по позициям)]`.


In [ ]:
f = clients[clients["acquisition_channel"] == "organic"]

answer = None   # ← ваш расчёт

check("★2", answer)


### ★ 3. Целое, которое переживёт пропуск

Слайд 15. У колонки `gap` из раздела 5 первое значение каждой группы — `NaT`:
предыдущего платежа не было.

Переведите `gap` в **целые минуты** так, чтобы пропуски остались пропусками,
а не превратились в ноль и не сломали тип. Обычный `astype("int64")` на этом
падает — нужен тип с поддержкой пропусков.

**Ответ:** список `[сколько значений не пропуск, сумма целых минут по непропускам]`,
оба числа целые.


In [ ]:
answer = None   # ← ваш расчёт

check("★3", answer)


### ★ 4. `iterrows` ломает типы

Слайды 22 и 23. Возьмите три числовые колонки и пять первых строк:

```python
nums = clients[["region_code", "birth_year", "kyc_level"]].head(5)
```

Достаньте `region_code` первой строки двумя способами: через `iterrows()`
и через `itertuples()`. Посмотрите на типы.

**Ответ:** список из двух строк — имена типов, `[через iterrows, через itertuples]`.
Имя типа берётся как `type(x).__name__`.


In [ ]:
nums = clients[["region_code", "birth_year", "kyc_level"]].head(5)

answer = None   # ← ваш расчёт

check("★4", answer)


### ★ 5. CSV не хранит типы

Слайд 16. Выгрузите тысячу платежей в CSV прямо в память — без файла на диске:

```python
import io

buf = io.StringIO()
payments.head(1000).to_csv(buf, index=False)
```

Прочитайте обратно и посчитайте, **сколько секунд** между самым ранним и самым
поздним `paid_at` в этой тысяче. Если прочитать наивно, вычитание упадёт: дата
вернётся текстом.

**Ответ:** одно целое число — секунды.


In [ ]:
import io

buf = io.StringIO()
payments.head(1000).to_csv(buf, index=False)

answer = None   # ← ваш расчёт

check("★5", answer)


---

## Что унести с сегодняшнего дня

1. **Дефект находят проверкой, которая знает, что должно быть правдой.** Взглядом
   на итоговую цифру не находят ничего: все четыре сегодняшних числа выглядели
   нормально.
2. **Величина сдвига ничего не говорит о цене дефекта.** 0,11 % — это 232 человека,
   с которых списали дважды.
3. **Инвариант связывает колонки, а не описывает одну.** «Кончилось не раньше, чем
   началось», «сумма больше нуля», «два списания подряд — не бывает».
4. **Пропуск исчезает молча.** `dropna=False` и `isna().sum()` рядом с каждой
   группировкой.
5. **`category` — не кнопка «сделать хорошо»**: смотрите на `nunique() / len()`.
6. **Проверка живёт в коде, а не в голове.** Четыре функции в `src/prime/checks.py`,
   и в ДЗ-2 за них 0,5 балла.

### Где вы запустили то, что было на лекции

| Слайд Л3 | Приём | Где |
|---|---|---|
| 27 | Семь вопросов приёмки | раздел 1 — это оглавление всего ноутбука |
| 10–11 | `memory_usage(deep=True)`, вес uuid | раздел 1 |
| 13–14 | `category`, правило `nunique() / len()` | раздел 1, вместе с ловушкой |
| 25 | `groupby` теряет пропуски, `dropna=False` | раздел 2 |
| 15 | Пропуск меняет тип, `Int64` | ★3 |
| 26 | `duplicated`, порог, почему `drop_duplicates` не годится | раздел 5 |
| 19–21 | `b = a` не копия, Copy-on-Write | на доске в начале пары |
| 8 | Индекс складывает за вас | ★1 |
| 7, 18 | `.loc` по метке против `.iloc` по позиции | ★2 |
| 22–23 | `iterrows` медленный и ломает типы | ★4 |
| 16 | Пять способов прочитать, CSV не хранит типы | ★5 |
| 12 | Деньги приехали не тем типом | семинар 4, раздел 1 |

### К следующей паре

Вторая пара сегодня — **витрина клиента**: соберём из шести таблиц одну, где одна
строка — один клиент. Там же выяснится, почему соединение двух таблиц умеет
создавать деньги из воздуха.

**ДЗ-1 — 27 сентября, 23:59.** Приём со снижением балла до 4 октября.
